# Agent Evaluation and Framework Trade-offs

Agent evaluation should inspect the trajectory (the sequence of tool calls the agent made), not only the final answer. A good final sentence can still hide a wrong tool call, missing evidence, or skipped human review.

In this notebook, you score the operations assistant on a small evaluation set, check what those scores can and cannot tell you, and compare agent frameworks.

## Learning Objectives

- Score completed tool calls, answer terms, and review requirements separately.
- Explain what scripted evaluation cases do and do not demonstrate.
- Choose an agent framework using operational constraints.

## Mental Model

Evaluating an agent works like testing code. Each case pairs a request with labels for the expected behavior. Running the case produces a trace, and separate checks compare it with each label, because each check catches a different failure:

- **Tool recall** catches a missing step, such as a skipped policy lookup.
- **Term recall** catches an answer that leaves out too many required facts.
- **Review match** catches a risky decision that skipped human review.

Passing all checks shows the agent still behaves as labeled: a regression test, not proof that its answers are correct.


```mermaid
flowchart LR
    Case["agent_cases.json<br/>E-02: critical incident T-1003"] --> Run["build_agent().run"]
    Run --> Trace["trace_from_result:<br/>completed tool calls"]
    Trace --> TR["Tool recall<br/>expected: ticket, policy, owner, human_review"]
    Trace --> AR["Required-term recall<br/>incident command, 60 minutes, Sam"]
    Trace --> RV["Request, ticket and review match<br/>including human_review arguments"]
    TR --> Verdict{"Regression checks pass?"}
    AR --> Verdict
    RV --> Verdict
```

Tool-use metrics need the same care as classification metrics: high recall can mean the agent found all required actions, while poor precision can mean it called extra tools unnecessarily.

<div align="center">
  <img src="assets/04-precision-recall.png" alt="Precision and recall diagram" width="400" style="max-width: 100%; height: auto;" />
  <p><em>Precision asks how many retrieved items are relevant; recall asks how many relevant items were retrieved. Source: <a href="https://commons.wikimedia.org/wiki/File:Precisionrecall.svg" target="_blank" rel="noopener noreferrer">Wikimedia Commons</a></em></p>
</div>

## Load The Shared Helpers


In [ ]:
import json
from pathlib import Path

from support import agent_limits, build_agent, evaluate_trace, trace_from_result

## Score The Evaluation Cases

The evaluation set labels expected tool use, required answer terms, and whether human review is required. `evaluate_trace` turns each label into a field of the result:

- `tool_recall`: the share of expected tools that appear in the trace.
- `term_recall`: the share of required terms found as whole words or phrases in the answer.
- `review_match`: the review flag matches the label, and a required review was called for the requested ticket.
- `request_match` and `ticket_match`: the trace belongs to the evaluated request and its ticket.

A case `passed` when every expected tool was called, at least two thirds of the required terms appear, and all match fields are `True`. Term matching is lexical, not semantic: an answer saying "do not escalate to Sam" still contains "Sam". Checking meaning needs a human reviewer or an LLM judge, which this notebook does not use.

In [ ]:
cases = json.loads(Path("data/evaluation/agent_cases.json").read_text())
cases

Run all supplied cases through PydanticAI, then score completed tool calls and structured outputs. In dry-run mode, this tests scripted orchestration. In live mode, it measures the selected model on these supplied cases only. Tool-name coverage and lexical term matches still do not prove semantic correctness. Each case starts without message history.

In [ ]:
agent = build_agent()
limits = agent_limits()
print("Model:", type(agent.model).__name__)
results = []
traces = []
for case in cases:
    result = await agent.run(case["request"], usage_limits=limits)
    trace = trace_from_result(result, case["request"], case_id=case["case_id"])
    traces.append(trace)
    results.append(evaluate_trace(trace, case))
results

Case `E-03` is different from the routing cases: it names no ticket, so the agent answers from durable memory with a single `recall_memory` call. Its trace and answer show what that single call returned:

In [ ]:
print([step.tool for step in traces[2].steps])
traces[2].decision.customer_update

Recall only asks whether the expected tools were called, so extra calls never lower it. Precision, the share of called tools that were expected, reveals them:

In [ ]:
for case, trace in zip(cases, traces):
    called = [step.tool for step in trace.steps]
    expected = set(case["expected_tools"])
    precision = (
        sum(tool in expected for tool in called) / len(called) if called else 0.0
    )
    print(case["case_id"], called, f"precision={precision:.2f}")

Both routing cases also called `recall_memory`, which their labels do not expect. Whether that is a problem depends on the labels: an extra read-only lookup is cheap, while an extra action could be costly. Precision makes the difference visible, so a human can decide.

Evaluation closes the loop: if the agent cannot show the right tool path, stay within its limits, and flag risky cases for review, a good final answer is not enough evidence to ship it.

## Exercise: Find What Each Check Catches

Each field of the evaluation result is meant to catch a different failure. Test that with damaged copies of the critical-case trace `traces[1]`:

- write `apply_regression(trace, kind)` that returns a copy of the trace with one regression: `"skipped review"` removes the `human_review` step, `"wrong ticket"` changes the decision's `ticket_id` to `T-1001`, and `"missing term"` replaces `every 60 minutes` in the customer update with `regularly`,
- evaluate each copy against `cases[1]` and print `tool_recall`, `term_recall`, `review_match`, `ticket_match`, and `passed`,
- for each regression, name the field that catches it. One regression passes anyway: explain why.

Use `dataclasses.replace` and `model_copy(update=...)`, so the original trace stays unchanged.

In [ ]:
# @TODO: Write apply_regression, then evaluate each damaged copy of traces[1].


<details>
<summary>Show Reference Solution</summary>

```python
from dataclasses import replace


def apply_regression(trace, kind):
    decision = trace.decision
    if kind == "skipped review":
        steps = [step for step in trace.steps if step.tool != "human_review"]
        return replace(trace, steps=steps)
    if kind == "wrong ticket":
        changed = decision.model_copy(update={"ticket_id": "T-1001"})
        return replace(trace, decision=changed)
    if kind == "missing term":
        update = decision.customer_update.replace("every 60 minutes", "regularly")
        changed = decision.model_copy(update={"customer_update": update})
        return replace(trace, decision=changed)
    raise ValueError(f"Unknown regression: {kind}")


fields = ["tool_recall", "term_recall", "review_match", "ticket_match", "passed"]
for kind in ["skipped review", "wrong ticket", "missing term"]:
    score = evaluate_trace(apply_regression(traces[1], kind), cases[1])
    print(kind, {field: score[field] for field in fields})
```

</details>


A skipped review is caught twice: `tool_recall` drops to `0.75` and `review_match` turns `False`. A decision for the wrong ticket only fails `ticket_match`. The missing term passes: two of three required terms still appear, which meets the two-thirds threshold. That tolerance is a design choice. It accepts an answer that phrases one fact differently, but it also lets a missing fact through. Every threshold trades false alarms against missed regressions, so labels and thresholds need the same review as the agent itself.

## Compare Agent Frameworks

Framework choice depends on constraints: the shape of the workflow (a linear tool loop or a graph with durable state), how strictly outputs must be validated, whether several specialist agents are really needed, and which stack the team already runs.

| Framework | Strength | Trade-off |
|---|---|---|
| **PydanticAI** | Typed tools, dependencies, and result validation | Less graph-native than LangGraph for complex state machines |
| **LangGraph** | Explicit graph state and durable orchestration | More framework surface for a short lesson |
| **Google ADK** | Vendor-native agent development and deployment path | Stronger fit when the target stack is Google-centered |
| **CrewAI** | Role-based crews and readable multi-agent demos | More coordination overhead when one agent would solve the task |

## Check Your Understanding

1. Why evaluate an agent's trajectory and not only its final answer?
2. An agent calls every expected tool plus one unnecessary tool. Which metric stays perfect, and which one reveals the extra call?
3. An agent passes every case in a small scripted evaluation set. What does that show, and what does it not show?
4. For a small support-routing workflow with strict schemas and a Python backend, which framework would you start with and why?

<details>
<summary>Show Reference Answers</summary>

1. A good final answer can hide a wrong tool call, missing evidence, or a skipped review. Only the trajectory shows how the agent reached its answer, and whether it took the steps that make the answer trustworthy.
2. Tool recall stays at `1.0`, because it only asks whether the expected tools were called. Tool precision drops, because one of the called tools was not expected: in case `E-01`, three of four calls were expected, so precision is `0.75`.
3. It shows that the agent still behaves as labeled on those cases: the expected tools run, the required terms appear, and reviews are flagged. It does not show that the answers are correct in meaning, because term matching is lexical, and with scripted replies it does not measure a live model's reasoning or how the agent handles requests outside the set.
4. Start with PydanticAI. The main requirement is typed tool calling and structured validation, not a large graph or role-playing crew. LangGraph becomes attractive when the workflow needs durable graph state; CrewAI becomes attractive when independent specialist roles are truly useful.

</details>


## Further Reading

- [LangSmith Evaluation Concepts](https://docs.langchain.com/langsmith/evaluation-concepts)
- [Google ADK Evaluation](https://google.github.io/adk-docs/evaluate/)
